# Proyecto 3: Matemáticas Híbridas Simbólico-Numéricas

**Objetivo**: Demostrar el flujo de trabajo matemático híbrido: Definir → Derivar simbólicamente → Evaluar numéricamente → Visualizar.

**Features demostradas**:
- `defmath` para definir funciones matemáticas una vez y obtener versiones simbólica y numérica
- `sympy-bridge` para derivación, integración y simplificación simbólica
- `numpy-fast` para evaluación vectorizada
- `defnp` para despacho automático escalar/vectorizado
- Exportación LaTeX de expresiones simbólicas
- Visualización con matplotlib

**Aplicaciones**:
- Gradiente y Hessiano de funciones multivariable
- Optimización con descenso de gradiente
- Series de Taylor
- Jacobiano de sistemas de ecuaciones

## 1. Inicialización

Cargamos las librerías pesadas (importan NumPy y SymPy).

In [ ]:
(load "lib/sympy-bridge.lp")
(load "lib/defmath.lp")
(load "lib/numpy-fast.lp")
(py:import "matplotlib.pyplot")
(py:import "numpy")

## 2. Funciones Matemáticas con defmath

`defmath` genera automáticamente:
- `nombre-sym`: versión simbólica (SymPy)
- `nombre-num`: versión numérica vectorizada (NumPy)

In [ ]:
;; Definir función híbrida: f(x) = sin(x) * exp(-x²/2)
(defmath onda-amortiguada (x)
  (* (sin x) (exp (* -0.5 (expt x 2)))))

;; Versión numérica — evaluar en 5 puntos
(let ((xs (numpy-linspace -4 4 5)))
  (format t "f(x) en 5 puntos:~%")
  (print (onda-amortiguada-num xs)))

;; Versión simbólica
(let ((x (sympy-symbol "x")))
  (format t "~%Expresión simbólica:~%")
  (print (onda-amortiguada-sym x)))

## 3. Derivación Simbólica

Derivamos la función y la visualizamos junto con su derivada.

In [ ]:
;; Derivar simbólicamente
(let* ((x (sympy-symbol "x"))
       (f (onda-amortiguada-sym x))
       (df (sympy-derive f x))
       (d2f (sympy-derive df x)))
  (format t "f(x)   = ~A~%" (sympy-simplify f))
  (format t "f'(x)  = ~A~%" (sympy-simplify df))
  (format t "f''(x) = ~A~%" (sympy-simplify d2f))
  (format t "~%LaTeX:~%")
  (format t "f(x) = ~A~%" (sympy-latex f))
  (format t "f'(x) = ~A~%" (sympy-latex df)))

## 4. Función Multivariable — Gradiente y Hessiano

Definimos g(x, y) = sin(x)·cos(y)·exp(-(x² + y²)/4) y calculamos su gradiente simbólico.

In [ ]:
(defmath campo-2d (x y)
  (* (sin x) (cos y) (exp (/ (- (+ (expt x 2) (expt y 2))) 4))))

;; Gradiente simbólico
(let* ((x (sympy-symbol "x"))
       (y (sympy-symbol "y"))
       (g (campo-2d-sym x y))
       (dgx (sympy-derive g x))
       (dgy (sympy-derive g y)))
  (format t "g(x,y) = ~A~%" (sympy-simplify g))
  (format t "∂g/∂x  = ~A~%" (sympy-latex (sympy-simplify dgx)))
  (format t "∂g/∂y  = ~A~%" (sympy-latex (sympy-simplify dgy))))

## 5. Visualización 3D de la Función

Evaluamos la función numéricamente en una grilla y visualizamos.

In [ ]:
(py:run "
import matplotlib.pyplot as plt
import numpy as np

# Crear grilla
xs = np.linspace(-4, 4, 100)
ys = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(xs, ys)

# Evaluar función numéricamente
Z = campo_2d_num(X, Y)

# Gráfico 3D
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9)
ax1.set_title('g(x,y) = sin(x)·cos(y)·exp(-(x²+y²)/4)')
ax1.set_xlabel('x')
ax1.set_ylabel('y')

ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contourf(X, Y, Z, levels=20, cmap='viridis')
ax2.set_title('Curvas de Nivel')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.savefig('campo-2d.png', dpi=100)
plt.show()
")

## 6. Optimización con Descenso de Gradiente

Implementamos descenso de gradiente usando la versión numérica de la función y su gradiente.

In [ ]:
;; Función de Rosenbrock con defmath
(defmath rosenbrock (x y)
  (+ (expt (- 1 x) 2) (* 100 (expt (- y (expt x 2)) 2))))

;; Obtener derivadas para gradiente
(let* ((x (sympy-symbol "x"))
       (y (sympy-symbol "y"))
       (f (rosenbrock-sym x y))
       (dfx (sympy-simplify (sympy-derive f x)))
       (dfy (sympy-simplify (sympy-derive f y)))
       (dfx-num (sympy-evalf dfx))
       (dfy-num (sympy-evalf dfy)))
  (format t "Rosenbrock: f(x,y) = (1-x)² + 100(y-x²)²~%")
  (format t "Evaluar en (2, 2): f = ~A~%" (rosenbrock-num 2 2)))

In [ ]:
;; Descenso de gradiente desde Lisp orquestando NumPy
(py:run "
import numpy as np

def rosenbrock_grad(x, y):
    dx = -2*(1 - x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

def rosenbrock_val(x, y):
    return (1 - x)**2 + 100*(y - x**2)**2

# Descenso de gradiente
punto = np.array([2.0, 2.0])
alpha = 0.001
trayectoria = [punto.copy()]
valores = [rosenbrock_val(*punto)]

for i in range(10000):
    grad = rosenbrock_grad(*punto)
    punto = punto - alpha * grad
    if i % 1000 == 0:
        trayectoria.append(punto.copy())
        valores.append(rosenbrock_val(*punto))

print(f'Punto inicial: (2.0, 2.0), f = {rosenbrock_val(2.0, 2.0):.2f}')
print(f'Punto final: ({punto[0]:.4f}, {punto[1]:.4f}), f = {rosenbrock_val(*punto):.6f}')
print(f'Mínimo teórico: (1, 1), f = 0')
")

In [ ]:
;; Visualizar optimización
(py:run "
fig, ax = plt.subplots(figsize=(10, 8))

# Curvas de nivel
x_vals = np.linspace(-0.5, 2.5, 200)
y_vals = np.linspace(-0.5, 4.5, 200)
X, Y = np.meshgrid(x_vals, y_vals)
Z = rosenbrock_num(X, Y)
levels = np.logspace(-1, 4, 25)
ax.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=0.6)

# Trayectoria
tray = np.array(trayectoria)
ax.plot(tray[:, 0], tray[:, 1], 'r.-', markersize=6, linewidth=2, label='Descenso')
ax.plot(1, 1, 'g*', markersize=15, label='Mínimo (1,1)')
ax.plot(2, 2, 'ro', markersize=8, label='Inicio (2,2)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Optimización: Descenso de Gradiente en Rosenbrock')
ax.legend()
ax.grid(True, alpha=0.3)

plt.savefig('optimizacion.png', dpi=100)
plt.show()
")

## 7. defnp — Despacho Automático Escalar/Vectorizado

`defnp` detecta automáticamente si los argumentos son escalares o arrays y despacha al modo adecuado.

In [ ]:
;; defnp: una función, dos modos
(defnp distancia-euclidiana (x y)
  (sqrt (+ (expt x 2) (expt y 2))))

;; Modo escalar
(format t "Escalar d(3, 4) = ~A~%" (distancia-euclidiana 3 4))

;; Modo vectorizado (con arrays NumPy)
(let ((xs (numpy-linspace 1 5 5))
      (ys (numpy-linspace 1 5 5)))
  (print (distancia-euclidiana xs ys)))

## 8. Integrales Definidas y Series de Taylor

Integración simbólica y expansión en series.

In [ ]:
;; Integral definida de sin(x²) de 0 a π
(let* ((x (sympy-symbol "x"))
       (expr (sympy-expr (sin (expt x 2))))
       (integral (sympy-integrate expr x)))
  (format t "∫ sin(x²) dx = ~A~%" integral)
  (format t "Valor numérico en [0, π]: ~A~%"
          (sympy-evalf integral x pi)))

;; Serie de Taylor de exp(sin(x)) alrededor de 0
(py:run "
import sympy as sp
x = sp.Symbol('x')
f = sp.exp(sp.sin(x))
taylor = sp.series(f, x, 0, 7).removeO()
print(f'exp(sin(x)) ≈ {taylor}')
print(f'LaTeX: {sp.latex(taylor)}')
")

## 9. Sistema de Ecuaciones — Jacobiano

Resolvemos un sistema no lineal usando el método de Newton con Jacobiano simbólico.

In [ ]:
(py:run "
import sympy as sp
import numpy as np

# Sistema: x² + y² = 4, x·y = 1
x, y = sp.symbols('x y')
f1 = x**2 + y**2 - 4
f2 = x*y - 1

# Jacobiano
F = sp.Matrix([f1, f2])
J = F.jacobian([x, y])

print('Sistema:')
print(f'  f1: {f1} = 0')
print(f'  f2: {f2} = 0')
print(f'\\nJacobiano J =')
sp.pprint(J)

# Newton-Raphson numérico
F_num = sp.lambdify((x, y), F, 'numpy')
J_num = sp.lambdify((x, y), J, 'numpy')

p = np.array([0.5, 2.0], dtype=float)
for i in range(10):
    F_val = np.array(F_num(*p)).flatten()
    if np.linalg.norm(F_val) < 1e-10:
        print(f'Convergencia en iteración {i}')
        break
    J_val = np.array(J_num(*p))
    delta = np.linalg.solve(J_val, -F_val)
    p = p + delta

print(f'Solución: x={p[0]:.8f}, y={p[1]:.8f}')
print(f'Verificación: f1={F_num(*p)[0,0]:.2e}, f2={F_num(*p)[1,0]:.2e}')
")

## 10. Ejercicios Propuestos

1. Implementar el método de Newton multivariable como macro `with-newton`
2. Crear `defmath` para la función de Himmelblau y encontrar sus 4 mínimos
3. Calcular la matriz Hessiana simbólica y verificar convexidad
4. Implementar integración por Monte Carlo desde Lisp usando NumPy
5. Crear una función que genere automáticamente el LaTeX de cualquier `defmath`